# Đọc và làm việc với file Parquet lớn (55.6 triệu dòng)

In [6]:
import os
import time
import pandas as pd
import pyarrow.parquet as pq

# Đường dẫn tới file Parquet đã gộp
PARQUET_PATH = r"D:\Download\E2E\E2E_NamNgu_Omachi_Wakeup_Mapping.parquet"

# 1. Xem Metadata của file (Không tải dữ liệu vào RAM, cực nhanh)
parquet_file = pq.ParquetFile(PARQUET_PATH)
print(f"Số dòng trong file Parquet: {parquet_file.metadata.num_rows:,}")
print(f"Số cột trong file Parquet: {parquet_file.metadata.num_columns}")
print("\nDanh sách các cột:")
print(parquet_file.schema.names)
print("\nChi tiết Schema:")
print(parquet_file.schema)

Số dòng trong file Parquet: 55,622,100
Số cột trong file Parquet: 17

Danh sách các cột:
['date', 'outlet_type', 'outlet_code', 'item_code', 'product_name', 'brand', 'dc', 'unit', 'forecast_qty', 'Factory_Code', 'Case', 'Factory', 'DC_Out', 'MD06[UOM1]', 'MD06[UOM2]', 'MD06[UOM Conversion]', 'MD06[Item Name]']

Chi tiết Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 date (String);
  optional binary field_id=-1 outlet_type (String);
  optional binary field_id=-1 outlet_code (String);
  optional binary field_id=-1 item_code (String);
  optional binary field_id=-1 product_name (String);
  optional binary field_id=-1 brand (String);
  optional binary field_id=-1 dc (String);
  optional binary field_id=-1 unit (String);
  optional double field_id=-1 forecast_qty;
  optional binary field_id=-1 Factory_Code (String);
  optional double field_id=-1 Case;
  optional binary field_id=-1 Factory (String);
  optional binary field_id=-1 DC_Out (String);
  optional binary fi

### 2. Đọc thử một vài dòng đầu tiên (Head)
Tải một lượng nhỏ dữ liệu để xem trước nội dung.

In [7]:
# Đọc 5 dòng đầu tiên

head_table = parquet_file.read_row_group(0).slice(0, 5)
df_head = head_table.to_pandas()
df_head

,date,outlet_type,outlet_code,item_code,product_name,brand,dc,unit,forecast_qty,Factory_Code,Case,Factory,DC_Out,MD06[UOM1],MD06[UOM2],MD06[UOM Conversion],MD06[Item Name]
0,2026-06-01,outlet,3203170,03NM00787,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml,Nam Ngư,MTN,chai,0.109844,null,0.036615,,DND,Blo,Thu,Thu,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml
1,2026-06-02,outlet,3203170,03NM00787,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml,Nam Ngư,MTN,chai,0.096662,null,0.032221,,DND,Blo,Thu,Thu,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml
2,2026-06-03,outlet,3203170,03NM00787,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml,Nam Ngư,MTN,chai,0.092269,null,0.030756,,DND,Blo,Thu,Thu,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml
3,2026-06-04,outlet,3203170,03NM00787,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml,Nam Ngư,MTN,chai,0.096662,null,0.032221,,DND,Blo,Thu,Thu,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml
4,2026-06-05,outlet,3203170,03NM00787,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml,Nam Ngư,MTN,chai,0.101056,null,0.033685,,DND,Blo,Thu,Thu,Nước mắm Nam Ngư (QR) 3bl x 8chai x 500ml


### 3. Chỉ đọc một số cột nhất định (Khuyên dùng để tiết kiệm RAM)
Vì file rất lớn (55 triệu dòng), nếu bạn chỉ cần phân tích một vài cột, hãy chỉ định tên cột khi đọc để tiết kiệm bộ nhớ.

In [ ]:
# Ví dụ chỉ đọc các cột: date, dc, forecast_qty, Factory_Code
columns_to_read = ['date', 'dc', 'forecast_qty', 'Factory_Code']
df_subset = pd.read_parquet(PARQUET_PATH, columns=columns_to_read)
print(f"Đã tải {len(df_subset):,} dòng với {df_subset.shape[1]} cột.")
df_subset.head()

### 4. Thống kê Số lượng Outlet, Item theo DC và Dòng Tổng cộng (Total)
Chỉ tải các cột cần thiết (`dc`, `outlet_code`, `item_code`) để thống kê:
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng mặt hàng thực tế duy nhất (`So_Item_Unique`)
- Tổng số dòng dữ liệu (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [4]:
start = time.time()

print("Đang đọc dữ liệu các cột 'dc', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...")
df_dc = pd.read_parquet(PARQUET_PATH, columns=['dc', 'outlet_code', 'item_code', 'Case', 'forecast_qty'])

print("Đang tính toán thống kê theo từng DC...")
grouped_dc = df_dc.groupby('dc').agg(
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_Item_Unique=('item_code', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

grouped_dc = grouped_dc.sort_values(by='So_Outlet_Unique', ascending=False).reset_index(drop=True)

print("Đang tính toán dòng tổng cộng (Total) theo DC...")
total_unique_outlets_dc = df_dc['outlet_code'].nunique()
total_unique_items_dc = df_dc['item_code'].nunique()
total_case_dc = df_dc['Case'].sum()
total_forecast_qty_dc = df_dc['forecast_qty'].sum()
total_rows_sum_dc = len(df_dc)

total_row_dc = pd.DataFrame([{
    'dc': 'Total',
    'So_Outlet_Unique': total_unique_outlets_dc,
    'So_Item_Unique': total_unique_items_dc,
    'Tong_Case': total_case_dc,
    'Tong_Forecast_Qty': total_forecast_qty_dc,
    'Tong_So_Dong': total_rows_sum_dc
}])

result_dc_df = pd.concat([grouped_dc, total_row_dc], ignore_index=True)
print(f"Thống kê DC hoàn tất trong {time.time() - start:.1f} giây.")

result_dc_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_Item_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
})

Đang đọc dữ liệu các cột 'dc', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...
Đang tính toán thống kê theo từng DC...
Đang tính toán dòng tổng cộng (Total) theo DC...
Thống kê DC hoàn tất trong 26.9 giây.


,dc,So_Outlet_Unique,So_Item_Unique,Tong_Case,Tong_Forecast_Qty,Tong_So_Dong
0,MTA,"82,881",25,"1,053,388.65","8,240,248.66","8,729,130"
1,MDQ,"81,181",22,"2,344,330.07","12,473,682.18","7,100,580"
2,MTH,"78,079",32,"1,747,955.57","13,689,249.06","9,992,940"
3,GMT,"54,317",39,"746,174.63","5,522,246.76","5,047,200"
4,MTF,"45,507",28,"2,276,421.65","11,080,596.35","4,541,250"
5,MTD,"42,300",25,"2,044,573.34","9,535,842.73","3,547,290"
6,MTV,"39,261",5,"554,801.35","2,219,205.41","1,503,090"
7,MTS,"37,282",23,"867,849.22","6,448,321.42","4,740,120"
8,MDV,"29,548",23,"475,121.60","4,243,537.84","3,273,990"
9,MDX,"27,127",29,"579,350.47","3,901,136.14","2,934,870"


### 4.1. Thống kê Số lượng Outlet, Item theo Factory_Code và Dòng Tổng cộng (Total)
Tương tự mục 4 nhưng group by `Factory_Code` thay vì `dc`:
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng mặt hàng thực tế duy nhất (`So_Item_Unique`)
- Tổng Case (`Tong_Case`)
- Tổng Forecast Qty (`Tong_Forecast_Qty`)
- Tổng số dòng dữ liệu (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [ ]:
start = time.time()

print("Dang doc du lieu cac cot 'Factory_Code', 'Factory', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...")
df_factory = pd.read_parquet(PARQUET_PATH, columns=['Factory_Code', 'Factory', 'outlet_code', 'item_code', 'Case', 'forecast_qty'])

print("Dang tinh toan thong ke theo tung Factory_Code...")
grouped_factory = df_factory.groupby('Factory_Code').agg(
    Factory_Name=('Factory', 'first'),
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_Item_Unique=('item_code', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

grouped_factory = grouped_factory.sort_values(by='So_Outlet_Unique', ascending=False).reset_index(drop=True)

print("Dang tinh toan dong tong cong (Total) theo Factory_Code...")
total_row_factory = pd.DataFrame([{
    'Factory_Code': 'Total',
    'Factory_Name': '',
    'So_Outlet_Unique': df_factory['outlet_code'].nunique(),
    'So_Item_Unique': df_factory['item_code'].nunique(),
    'Tong_Case': df_factory['Case'].sum(),
    'Tong_Forecast_Qty': df_factory['forecast_qty'].sum(),
    'Tong_So_Dong': len(df_factory)
}])

result_factory_df = pd.concat([grouped_factory, total_row_factory], ignore_index=True)
print(f"Thong ke Factory_Code hoan tat trong {time.time() - start:.1f} giay.")

result_factory_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_Item_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
})

### 5. Thống kê Số lượng Outlet, Item theo Unit và Dòng Tổng cộng (Total)
Chỉ tải các cột cần thiết (`unit`, `outlet_code`, `item_code`) để thống kê:
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng mặt hàng thực tế duy nhất (`So_Item_Unique`)
- Tổng số dòng dữ liệu (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [5]:
start = time.time()

print("Đang đọc dữ liệu các cột 'unit', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...")
df_unit = pd.read_parquet(PARQUET_PATH, columns=['unit', 'outlet_code', 'item_code', 'Case', 'forecast_qty'])

print("Đang tính toán thống kê theo từng Unit...")
grouped_unit = df_unit.groupby('unit').agg(
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_Item_Unique=('item_code', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

grouped_unit = grouped_unit.sort_values(by='So_Outlet_Unique', ascending=False).reset_index(drop=True)

print("Đang tính toán dòng tổng cộng (Total) theo Unit...")
total_unique_outlets_unit = df_unit['outlet_code'].nunique()
total_unique_items_unit = df_unit['item_code'].nunique()
total_case_unit = df_unit['Case'].sum()
total_forecast_qty_unit = df_unit['forecast_qty'].sum()
total_rows_sum_unit = len(df_unit)

total_row_unit = pd.DataFrame([{
    'unit': 'Total',
    'So_Outlet_Unique': total_unique_outlets_unit,
    'So_Item_Unique': total_unique_items_unit,
    'Tong_Case': total_case_unit,
    'Tong_Forecast_Qty': total_forecast_qty_unit,
    'Tong_So_Dong': total_rows_sum_unit
}])

result_unit_df = pd.concat([grouped_unit, total_row_unit], ignore_index=True)
print(f"Thống kê Unit hoàn tất trong {time.time() - start:.1f} giây.")

result_unit_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_Item_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
})

Đang đọc dữ liệu các cột 'unit', 'outlet_code', 'item_code', 'Case', 'forecast_qty'...
Đang tính toán thống kê theo từng Unit...
Đang tính toán dòng tổng cộng (Total) theo Unit...
Thống kê Unit hoàn tất trong 30.0 giây.


,unit,So_Outlet_Unique,So_Item_Unique,Tong_Case,Tong_Forecast_Qty,Tong_So_Dong
0,chai,"424,537",18,"10,959,909.96","42,221,587.13","24,840,060"
1,goi,"244,881",3,"787,704.27","23,631,128.06","13,930,620"
2,cha,"233,536",18,"669,702.26","10,684,337.10","10,111,230"
3,lon,"113,151",3,"345,191.79","1,380,767.17","3,408,660"
4,can,"34,305",1,"80,130.32","320,521.29","1,038,570"
5,CHA,"4,958",11,"82,439.12","593,401.60","1,588,140"
6,G1,"4,957",3,"182,694.87","5,480,846.07","456,930"
7,LON,"4,951",1,"14,398.72","57,594.89","152,190"
8,hu,"2,962",2,"4,681.12","14,043.36","89,340"
9,CAN,117,1,"16,847.16","67,388.63","3,810"


### 6. Thống kê Số lượng Outlet, DC theo Item và Dòng Tổng cộng (Total)
Chỉ tải các cột cần thiết (`item_code`, `outlet_code`, `dc`) để thống kê theo từng sản phẩm (`item_code`):
- Số lượng Outlet thực tế duy nhất (`So_Outlet_Unique`)
- Số lượng DC thực tế phân phối mặt hàng này (`So_DC_Unique`)
- Tổng số dòng giao dịch của mặt hàng (`Tong_So_Dong`)
- Dòng tổng cộng (Total) ở dưới cùng.
- Định dạng hiển thị phân tách hàng ngàn bằng dấu phẩy `,`.

In [8]:
start = time.time()

print("Dang doc du lieu cac cot can thiet cho Item Stats...")
df_item = pd.read_parquet(PARQUET_PATH, columns=[
    'item_code', 'unit', 'outlet_code', 'dc',
    'Case', 'forecast_qty',
    'MD06[UOM1]', 'MD06[UOM2]', 'MD06[UOM Conversion]', 'MD06[Item Name]'
])

print("Dang tinh toan thong ke theo cap (item_code, unit)...")
grouped_item = df_item.groupby(['item_code', 'unit']).agg(
    Item_Name=('MD06[Item Name]', 'first'),
    UOM1=('MD06[UOM1]', 'first'),
    UOM2=('MD06[UOM2]', 'first'),
    UOM_Conversion=('MD06[UOM Conversion]', 'first'),
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_DC_Unique=('dc', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

# Sap xep giam dan theo tong so dong giao dich
grouped_item = grouped_item.sort_values(by=['item_code', 'Tong_So_Dong'], ascending=[True, False]).reset_index(drop=True)

print("Dang tinh toan dong tong cong (Total)...")
total_row_item = pd.DataFrame([{
    'item_code': 'Total',
    'unit': '',
    'Item_Name': '',
    'UOM1': '',
    'UOM2': '',
    'UOM_Conversion': '',
    'So_Outlet_Unique': df_item['outlet_code'].nunique(),
    'So_DC_Unique': df_item['dc'].nunique(),
    'Tong_Case': df_item['Case'].sum(),
    'Tong_Forecast_Qty': df_item['forecast_qty'].sum(),
    'Tong_So_Dong': len(df_item)
}])

result_item_df = pd.concat([grouped_item, total_row_item], ignore_index=True)
print(f"Hoan tat trong {time.time() - start:.1f} giay. So cap (item_code, unit): {len(grouped_item):,}")

# Hien thi bang
display(result_item_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_DC_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
}))

# Xuat ra file Excel
excel_out = r"D:/Download/E2E/Item_Stats_Summary.xlsx"
print(f"Dang xuat ra Excel: {excel_out}")
result_item_df.to_excel(excel_out, index=False, engine='openpyxl')
print(f"Da xuat xong file Excel: {excel_out}")

Dang doc du lieu cac cot can thiet cho Item Stats...
Dang tinh toan thong ke theo cap (item_code, unit)...
Dang tinh toan dong tong cong (Total)...
Hoan tat trong 127.5 giay. So cap (item_code, unit): 64


,item_code,unit,Item_Name,UOM1,UOM2,UOM_Conversion,So_Outlet_Unique,So_DC_Unique,Tong_Case,Tong_Forecast_Qty,Tong_So_Dong
0,02OM00338,goi,Mì dinh dưỡng khoai tây Omachi Sườn hầm ngũ quả 30gói x 80gr,Goi,Thu,Thu,"186,716",10,"365,396.05","10,961,881.37","5,627,820"
1,02OM00338,G1,Mì dinh dưỡng khoai tây Omachi Sườn hầm ngũ quả 30gói x 80gr,Goi,Thu,Thu,"4,954",7,"83,667.28","2,510,018.53","152,280"
2,02OM00618,goi,Mì dinh dưỡng khoai tây Omachi mì trộn xốt Spaghetti 30gói x 90gr,Goi,Thu,Thu,"110,164",10,"135,494.96","4,064,848.85","3,331,440"
3,02OM00618,G1,Mì dinh dưỡng khoai tây Omachi mì trộn xốt Spaghetti 30gói x 90gr,Goi,Thu,Thu,"4,955",7,"30,689.52","920,685.59","152,310"
4,02OM00770,goi,Mì dinh dưỡng khoai tây Omachi xốt Bò hầm 30gói x 81gr,Goi,Thu,Thu,"164,832",10,"286,813.26","8,604,397.83","4,971,360"
5,02OM00770,G1,Mì dinh dưỡng khoai tây Omachi xốt Bò hầm 30gói x 81gr,Goi,Thu,Thu,"4,956",7,"68,338.06","2,050,141.95","152,340"
6,03HH00033,chai,Combo Nam Ngư 12bộ x (1chai NC Nam Ngư Đệ Nhị 900ml + 1chai NM Nam Ngư 300ml),Bo,Thu,Thu,1,1,0.11,1.33,30
7,03NM00523,can,NC Nam Ngư Siêu tiết kiệm 4can x 4.8lít,Can,Thu,Thu,"34,305",10,"80,130.32","320,521.29","1,038,570"
8,03NM00523,CAN,NC Nam Ngư Siêu tiết kiệm 4can x 4.8lít,Can,Thu,Thu,117,7,"16,847.16","67,388.63","3,810"
9,03NM00532,cha,Nước mắm Nam Ngư cao cấp 15chai x 500ml,Cha,Thu,Thu,1,1,0.20,3.00,30


Dang xuat ra Excel: D:/Download/E2E/Item_Stats_Summary.xlsx
Da xuat xong file Excel: D:/Download/E2E/Item_Stats_Summary.xlsx


In [7]:
start = time.time()

print("Dang doc du lieu cac cot can thiet cho Item Stats...")
df_item = pd.read_parquet(PARQUET_PATH, columns=[
    'item_code', 'unit', 'outlet_code', 'dc',
    'Case', 'forecast_qty',
    'MD06[UOM1]', 'MD06[UOM2]', 'MD06[UOM Conversion]', 'MD06[Item Name]'
])

print("Dang tinh toan thong ke theo cap (item_code, unit)...")
grouped_item = df_item.groupby(['item_code', 'unit']).agg(
    Item_Name=('MD06[Item Name]', 'first'),
    UOM1=('MD06[UOM1]', 'first'),
    UOM2=('MD06[UOM2]', 'first'),
    UOM_Conversion=('MD06[UOM Conversion]', 'first'),
    So_Outlet_Unique=('outlet_code', 'nunique'),
    So_DC_Unique=('dc', 'nunique'),
    Tong_Case=('Case', 'sum'),
    Tong_Forecast_Qty=('forecast_qty', 'sum'),
    Tong_So_Dong=('outlet_code', 'count')
).reset_index()

# Sap xep giam dan theo tong so dong giao dich
grouped_item = grouped_item.sort_values(by=['item_code', 'Tong_So_Dong'], ascending=[True, False]).reset_index(drop=True)

print("Dang tinh toan dong tong cong (Total)...")
total_row_item = pd.DataFrame([{
    'item_code': 'Total',
    'unit': '',
    'Item_Name': '',
    'UOM1': '',
    'UOM2': '',
    'UOM_Conversion': '',
    'So_Outlet_Unique': df_item['outlet_code'].nunique(),
    'So_DC_Unique': df_item['dc'].nunique(),
    'Tong_Case': df_item['Case'].sum(),
    'Tong_Forecast_Qty': df_item['forecast_qty'].sum(),
    'Tong_So_Dong': len(df_item)
}])

result_item_df = pd.concat([grouped_item, total_row_item], ignore_index=True)
print(f"Hoan tat trong {time.time() - start:.1f} giay. So cap (item_code, unit): {len(grouped_item):,}")

# Hien thi bang
display(result_item_df.style.format({
    'So_Outlet_Unique': '{:,}',
    'So_DC_Unique': '{:,}',
    'Tong_Case': '{:,.2f}',
    'Tong_Forecast_Qty': '{:,.2f}',
    'Tong_So_Dong': '{:,}'
}))

# Xuat ra file Excel
excel_out = r"D:/Download/E2E/Item_Stats_Summary.xlsx"
print(f"Dang xuat ra Excel: {excel_out}")
result_item_df.to_excel(excel_out, index=False, engine='openpyxl')
print(f"Da xuat xong file Excel: {excel_out}")

Đang đọc dữ liệu các cột 'date', 'outlet_code', 'dc', 'Case', 'forecast_qty'...
Đang tính toán thống kê theo từng Item...
Đang tính toán dòng tổng cộng (Total) theo Item...
Thống kê Item hoàn tất trong 38.1 giây.


,date,So_Outlet_Unique,So_DC_Unique,Tong_Case,Tong_Forecast_Qty,Tong_So_Dong
0,2026-06-01,"492,307",16,"539,019.06","3,429,579.87","1,854,070"
1,2026-06-02,"492,307",16,"475,523.28","3,042,830.47","1,854,070"
2,2026-06-29,"492,307",16,"539,019.06","3,429,579.87","1,854,070"
3,2026-06-28,"492,307",16,"94,548.39","722,331.88","1,854,070"
4,2026-06-27,"492,307",16,"496,688.36","3,171,745.98","1,854,070"
5,2026-06-26,"492,307",16,"496,688.36","3,171,745.98","1,854,070"
6,2026-06-25,"492,307",16,"475,523.28","3,042,830.47","1,854,070"
7,2026-06-24,"492,307",16,"454,357.78","2,913,912.37","1,854,070"
8,2026-06-23,"492,307",16,"475,523.28","3,042,830.47","1,854,070"
9,2026-06-22,"492,307",16,"539,019.06","3,429,579.87","1,854,070"
